In [0]:
-- =====================================================================
-- CineClub Wrapped 2026
-- =====================================================================

-- Population: active annual / gift members + monthly upgraders (yesterday's snapshot)
CREATE OR REPLACE TEMPORARY VIEW active_CC_Users AS (
    SELECT DISTINCT
        cc_user_profile_id,
        cde_id,
        plan_id,
        renewal_status
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE to_date(snapshot_date_id, 'yyyyMMdd') = curdate() - 1
      AND (
            (plan_id IN ('annual1', 'gift-plan') AND renewal_status IN ('Active', 'Active_Non_Renewing'))
         OR (plan_id IN ('monthly1') AND renewal_status IN ('Active_Annual_Upgrade'))
      )
);

-- All 2026 transactions (visit date Jan 1 → yesterday)
CREATE OR REPLACE TEMPORARY VIEW cc_all_trans AS (
    SELECT
        cc.cde_id,
        transactiondateid,
        visitdateid,
        trans.LocationId,
        loc.Location_Name,
        loc.LC_Location_Type_Description,
        loc.LC_Trade_Brand_Description,
        trans.film_id,
        film.title_id,
        film.title_name,
        trans.sessionskey,
        transaction_number,
        local_item_id,
        item.Item_ID,
        item.item_description,
        TicketFLAG,
        ConcessionFLAG,
        Box_Office_SundryFLAG,
        COTFLAG,
        STOREFLAG,
        SCENEFLAG,
        CINECLUBFLAG,
        trans.IS_DBOX_Flag AS DBOXFLAG,
        CineClub_Discount_Type_ID,
        Quantity,
        Net_Spend,
        Net_Revenue
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction AS trans
    INNER JOIN active_CC_Users AS cc
        ON trans.cde_id = cc.cde_id
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_location AS loc
        ON trans.LocationId = loc.LocationID
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_film AS film
        ON trans.film_id = film.film_id
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_item AS item
        ON trans.ItemSkey = item.itemskey
    WHERE VisitDateId BETWEEN '20260101' AND date_format(curdate() - 1, 'yyyyMMdd')
);

-- Visit grain: one row per cde_id × date × location, refunds netted
CREATE OR REPLACE TEMPORARY VIEW cc_visits AS (
    SELECT
        cde_id,
        visitdateid,
        LocationId,
        LC_Location_Type_Description,
        SUM(Quantity) AS net_qty
    FROM cc_all_trans
    GROUP BY cde_id, visitdateid, LocationId, LC_Location_Type_Description
    HAVING SUM(Quantity) > 0
);

-- Wrapped population: active members with ≥1 visit in window
CREATE OR REPLACE TEMPORARY VIEW wrapped_members AS (
    SELECT DISTINCT cde_id
    FROM cc_visits
);

-- Viewing grain: member × date × location × film, net tickets > 0
CREATE OR REPLACE TEMPORARY VIEW cc_viewings AS (
    SELECT cde_id, visitdateid, LocationId, film_id, title_id
    FROM cc_all_trans
    WHERE TicketFLAG = 'Yes'
    GROUP BY cde_id, visitdateid, LocationId, film_id, title_id
    HAVING SUM(Quantity) > 0
);

-- Viewing-level session attributes
CREATE OR REPLACE TEMPORARY VIEW cc_viewing_formats AS (
    SELECT
        v.cde_id,
        v.visitdateid,
        v.LocationId,
        v.film_id,
        MAX(s.Auditorium_Experience_ID)  AS auditorium_id,
        MAX(s.Performance_Experience_ID) AS performance_id,
        MAX(t.DBOXFLAG)                  AS is_dbox,   -- DBOX from transaction table
        MAX(s.Is_70mmflag)               AS is_70mm,
        MAX(s.Is_IMAXflag)               AS is_imax
    FROM cc_viewings AS v
    INNER JOIN cc_all_trans AS t
        ON  v.cde_id      = t.cde_id
        AND v.visitdateid = t.visitdateid
        AND v.LocationId  = t.LocationId
        AND v.film_id     = t.film_id
        AND t.TicketFLAG  = 'Yes'
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_session AS s
        ON t.sessionskey = s.sessionskey
    GROUP BY v.cde_id, v.visitdateid, v.LocationId, v.film_id
);

-- Viewing-level experience label (V5 hierarchy + 70MM ahead of IMAX)
CREATE OR REPLACE TEMPORARY VIEW cc_viewing_experience AS (
    SELECT
        cde_id,
        visitdateid,
        LocationId,
        film_id,
        CASE
            WHEN is_dbox = 'Yes'                   THEN 'DBOX'
            WHEN is_70mm = 'Yes'                   THEN '70MM'
            WHEN auditorium_id = 2                 THEN 'AVX'
            WHEN auditorium_id = 3                 THEN 'IMAX'
            WHEN auditorium_id = 4                 THEN 'VIP'
            WHEN auditorium_id = 7                 THEN 'FourDX'
            WHEN auditorium_id = 8                 THEN 'ScreenX'
            WHEN performance_id = 2                THEN 'ThreeD'
            ELSE 'Regular'
        END AS final_format
    FROM cc_viewing_formats
);

-- Line level: kernels and ml per concession line
CREATE OR REPLACE TEMPORARY VIEW cc_pop_drink_lines AS (
    SELECT
        cde_id,
        item_description,
        CineClub_Discount_Type_ID,
        Quantity,
        Net_Revenue,
        Quantity * CASE
            WHEN item_description = 'VHO CORN - KIDDIE TRAY (SANS DRINK)'                        THEN 300
            WHEN item_description = 'VHO CORN - UPSIZE TO REGULAR CORN'                          THEN 100
            WHEN item_description IN ('VHO CORN - UPSIZE TO LARGE CORN',
                                      'VHO DELIVERY - POPCORN - UPSIZE TO LARGE CORN')           THEN 100
            WHEN item_description = 'VHO CORN - UPSIZE SMALL TO LARGE CORN'                      THEN 200
            WHEN item_description LIKE '%UPSIZE%'                                                THEN 0
            WHEN item_description LIKE '%POPCORN%LARGE%' OR item_description LIKE '%LARGE%POPCORN%'
              OR item_description LIKE '%VHO%CORN%VIP%POPCORN%LARGE%'                           THEN 500
            WHEN item_description LIKE '%POPCORN%REGULAR%' OR item_description LIKE '%REGULAR%POPCORN%'
              OR item_description LIKE '%VHO%CORN%VIP%POPCORN%REGULAR%'                         THEN 400
            WHEN item_description LIKE '%POPCORN%SMALL%' OR item_description LIKE '%SMALL%POPCORN%'
              OR item_description LIKE '%VHO%CORN%VIP%POPCORN%SMALL%'                           THEN 300
            ELSE 0 END AS kernels,
        Quantity * CASE
            WHEN item_description = 'VHO DRINK - UPSIZE LRG FOUNTAIN'                            THEN 150
            WHEN item_description LIKE '%UPSIZE SMALL FOUNTAIN%XL%'
              OR item_description LIKE '%UPSIZE REGULAR FOUNTAIN%XL%'
              OR item_description LIKE '%UPSIZE LARGE FOUNTAIN%XL%'                              THEN 500
            WHEN item_description LIKE '%VHO%DRINK%CUP%LARGE%'
              OR item_description LIKE '%VHO%DRINK%VIP%DRINK%LARGE%'                             THEN 500
            WHEN item_description LIKE '%VHO%DRINK%CUP%REGULAR%'
              OR item_description LIKE '%VHO%DRINK%VIP%DRINK%REGULAR%'                           THEN 350
            WHEN item_description LIKE '%VHO%DRINK%CUP%SMALL%'                                   THEN 250
            ELSE 0 END AS drink_ml
    FROM cc_all_trans
    WHERE ConcessionFLAG = 'Yes'
      AND LC_Location_Type_Description = 'Cineplex Theatre'
      AND STOREFLAG = 'No'
);

-- =====================================================================
-- Columns
-- =====================================================================

-- Column: CineClub join date
CREATE OR REPLACE TEMPORARY VIEW col_join_date AS (
    SELECT
        a.cde_id,
        to_date(
            from_utc_timestamp(
                timestamp_seconds(
                    COALESCE(
                        MIN(CASE WHEN e.event_type = 'customer_created' THEN e.record_occurred_at END),
                        MIN(e.record_occurred_at)
                    )
                ),
                'America/New_York'
            )
        ) AS joined_date
    FROM wrapped_members AS w
    INNER JOIN active_CC_Users AS a
        ON w.cde_id = a.cde_id
    LEFT JOIN cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
        ON a.cc_user_profile_id = e.cc_user_profile_id
    GROUP BY a.cde_id
);

-- Column: Total visits (theatre)
CREATE OR REPLACE TEMPORARY VIEW col_total_visits AS (
    SELECT
        cde_id,
        COUNT(*) AS total_visits
    FROM cc_visits
    WHERE LC_Location_Type_Description = 'Cineplex Theatre'
    GROUP BY cde_id
);

-- Column: Movie visits (visits with net tickets > 0)
CREATE OR REPLACE TEMPORARY VIEW col_movie_visits AS (
    SELECT
        cde_id,
        COUNT(*) AS movie_visits
    FROM (
        SELECT cde_id, visitdateid, LocationId
        FROM cc_all_trans
        WHERE TicketFLAG = 'Yes'
        GROUP BY cde_id, visitdateid, LocationId
        HAVING SUM(Quantity) > 0
    ) AS ticket_visits
    GROUP BY cde_id
);

-- Column: Concession visits (visits with net concession qty > 0)
CREATE OR REPLACE TEMPORARY VIEW col_concession_visits AS (
    SELECT
        cde_id,
        COUNT(*) AS concession_visits
    FROM (
        SELECT cde_id, visitdateid, LocationId
        FROM cc_all_trans
        WHERE ConcessionFLAG = 'Yes'
          AND LC_Location_Type_Description = 'Cineplex Theatre'
        GROUP BY cde_id, visitdateid, LocationId
        HAVING SUM(Quantity) > 0
    ) AS concession_visit_keys
    GROUP BY cde_id
);

-- Columns: Movies watched (incl. repeats) and unique movies watched
CREATE OR REPLACE TEMPORARY VIEW col_movies_watched AS (
    SELECT
        cde_id,
        COUNT(*)                 AS movies_watched,
        COUNT(DISTINCT title_id) AS unique_movies_watched
    FROM cc_viewings
    GROUP BY cde_id
);

-- Columns: Max rewatch count and its title (tie → most recent)
CREATE OR REPLACE TEMPORARY VIEW col_max_rewatch AS (
    WITH title_counts AS (
        SELECT
            cde_id,
            title_id,
            COUNT(*)         AS title_viewings,
            MAX(visitdateid) AS last_viewed
        FROM cc_viewings
        WHERE title_id IS NOT NULL
        GROUP BY cde_id, title_id
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY cde_id
                ORDER BY title_viewings DESC, last_viewed DESC, title_id
            ) AS rn
        FROM title_counts
    )
    SELECT
        cde_id,
        title_viewings AS max_rewatch_count,
        title_id       AS max_rewatch_title_id
    FROM ranked
    WHERE rn = 1
);

-- Column: Deduplicated list of all film_ids watched, ordered by first visitdateid
CREATE OR REPLACE TEMPORARY VIEW col_titles_watched AS (
    WITH first_seen AS (
        SELECT cde_id, film_id, MIN(visitdateid) AS first_visit
        FROM cc_viewings
        WHERE film_id IS NOT NULL
        GROUP BY cde_id, film_id
    )
    SELECT
        cde_id,
        transform(
            array_sort(collect_list(struct(first_visit, film_id))),
            x -> x.film_id
        ) AS titles_watched
    FROM first_seen
    GROUP BY cde_id
);

-- Column: Total hours watched (runtime × every viewing, rewatches included)
CREATE OR REPLACE TEMPORARY VIEW col_hours_watched AS (
    SELECT
        v.cde_id,
        ROUND(SUM(f.film_runtime) / 60.0, 1) AS total_hours_watched
    FROM cc_viewings AS v
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_film AS f
        ON v.film_id = f.film_id
    GROUP BY v.cde_id
);

-- Columns: CineClub savings 2026 (owner savings logic, adapted to the Wrapped base)
CREATE OR REPLACE TEMPORARY VIEW col_savings AS (
    WITH price_wide AS (
        SELECT * FROM VALUES
            ('1',   15.75, 16.00), ('2',   15.25, 15.50), ('3',   14.75, 15.00),
            ('4',   13.75, 14.00), ('5',   13.75, 14.00), ('6',   12.75, 13.00),
            ('8',   15.75, 16.00), ('9',   14.25, 14.50), ('11',  12.25, 12.50),
            ('12',  12.75, 13.00), ('14',  14.00, 14.25), ('15',  13.25, 13.50),
            ('16',  12.50, 12.75), ('17',  12.75, 13.00), ('18A', 13.75, 14.00),
            ('19',  11.25, 11.50), ('21',  15.25, 15.50), ('21J', 16.00, 16.25),
            ('22',  14.50, 14.75), ('23',  13.50, 13.75), ('24',  12.50, 12.75),
            ('24A', 14.50, 14.75), ('25',  12.25, 12.50), ('26',  13.75, 14.00),
            ('26A', 14.25, 14.50), ('27',  13.25, 13.50), ('28',  12.25, 12.50),
            ('29',  11.25, 11.50), ('30',  12.25, 12.50), ('31',  13.25, 13.50),
            ('32',  12.75, 13.00), ('33',  12.25, 12.50), ('34',  11.25, 11.50)
        AS t(Price_Category, price_oct25, price_nov26)
    ),
    price_category_dict AS (
        SELECT Price_Category, DATE'2025-10-01' AS Start_Date, DATE'2026-10-31' AS End_Date,
               CAST(price_oct25 AS DECIMAL(18,2)) AS Adult_General
        FROM price_wide
        UNION ALL
        SELECT Price_Category, DATE'2026-11-01', DATE'9999-12-31',
               CAST(price_nov26 AS DECIMAL(18,2))
        FROM price_wide
    ),
    price_hist AS (
        SELECT l.locationid AS LocationId, d.Adult_General, d.Start_Date, d.End_Date
        FROM data_exploration_hub.marketing_hub.mkt_pricing_tier_by_loc AS l
        JOIN price_category_dict AS d
          ON UPPER(TRIM(l.pricing_tier)) = d.Price_Category
    ),
    txn AS (
        SELECT
            cde_id, visitdateid, LocationId, Net_Revenue, Quantity,
            TicketFLAG, ConcessionFLAG, COTFLAG, CineClub_Discount_Type_ID,
            to_date(CAST(visitdateid AS STRING), 'yyyyMMdd') AS visit_date
        FROM cc_all_trans
        WHERE CINECLUBFLAG = 'Yes'
    ),
    price_lookup AS (
        SELECT d.LocationId, d.visit_date, p.Adult_General
        FROM (SELECT DISTINCT LocationId, visit_date FROM txn) AS d
        JOIN price_hist AS p
          ON  d.LocationId = p.LocationId
          AND d.visit_date BETWEEN p.Start_Date AND p.End_Date
        QUALIFY ROW_NUMBER() OVER (PARTITION BY d.LocationId, d.visit_date ORDER BY p.Start_Date DESC) = 1
    ),
    member_savings AS (
        SELECT
            t.cde_id,
            ROUND(SUM(CASE WHEN t.CineClub_Discount_Type_ID IN (5, 6)
                           THEN t.Quantity * (COALESCE(pl.Adult_General, 9.99) - 9.99) ELSE 0 END), 2)   AS ticket_savings_999,
            ROUND(SUM(CASE WHEN t.CineClub_Discount_Type_ID IN (7, 8)
                           THEN t.Quantity * (COALESCE(pl.Adult_General, 10.99) - 10.99) ELSE 0 END), 2) AS ticket_savings_1099,
            ROUND(SUM(CASE WHEN t.ConcessionFLAG = 'Yes' AND t.CineClub_Discount_Type_ID = 2
                           THEN t.Net_Revenue * 0.25 ELSE 0 END), 2)                                      AS concession_savings,
            ROUND(SUM(CASE WHEN t.ConcessionFLAG = 'Yes' AND t.CineClub_Discount_Type_ID = 3
                           THEN t.Net_Revenue * 0.25 ELSE 0 END), 2)                                      AS gaming_savings,
            ROUND(SUM(CASE WHEN t.TicketFLAG = 'Yes' AND t.COTFLAG = 'Yes'
                            AND t.CineClub_Discount_Type_ID IN (5, 6, 7, 8)
                           THEN t.Quantity * 1.50 ELSE 0 END), 2)                                         AS online_booking_savings
        FROM txn AS t
        LEFT JOIN price_lookup AS pl
          ON t.LocationId = pl.LocationId AND t.visit_date = pl.visit_date
        GROUP BY t.cde_id
    )
    SELECT
        cde_id,
        ticket_savings_999,
        ticket_savings_1099,
        concession_savings,
        gaming_savings,
        online_booking_savings,
        GREATEST(ROUND(ticket_savings_999 + ticket_savings_1099 + concession_savings
                       + gaming_savings + online_booking_savings, 2), 0) AS total_savings
    FROM member_savings
);

-- Columns: Total kernels and ml
CREATE OR REPLACE TEMPORARY VIEW col_pop_drink AS (
    SELECT
        cde_id,
        CAST(SUM(kernels)  AS INT) AS Total_Kernels,
        CAST(SUM(drink_ml) AS INT) AS Total_Drink_ml
    FROM cc_pop_drink_lines
    GROUP BY cde_id
);

-- Columns: Free kernels and free ml from the CineClub 20% discount
CREATE OR REPLACE TEMPORARY VIEW col_pop_drink_free AS (
    WITH disc AS (
        SELECT
            cde_id,
            SUM(CASE WHEN kernels  <> 0 THEN kernels END)                AS disc_kernels,
            SUM(CASE WHEN kernels  <> 0 THEN Net_Revenue / 0.8 END)      AS popcorn_full_price,
            SUM(CASE WHEN kernels  <> 0 THEN Net_Revenue * 0.25 END)     AS popcorn_savings,
            SUM(CASE WHEN drink_ml <> 0 THEN drink_ml END)               AS disc_ml,
            SUM(CASE WHEN drink_ml <> 0 THEN Net_Revenue / 0.8 END)      AS drink_full_price,
            SUM(CASE WHEN drink_ml <> 0 THEN Net_Revenue * 0.25 END)     AS drink_savings
        FROM cc_pop_drink_lines
        WHERE CineClub_Discount_Type_ID = 2
        GROUP BY cde_id
    )
    SELECT
        cde_id,
        ROUND(popcorn_savings, 2)                                       AS popcorn_savings,
        ROUND(drink_savings, 2)                                         AS drink_savings,
        disc_kernels / NULLIF(popcorn_full_price, 0)                    AS kernels_per_dollar,
        disc_ml      / NULLIF(drink_full_price, 0)                      AS ml_per_dollar,
        CAST(ROUND(popcorn_savings * disc_kernels / NULLIF(popcorn_full_price, 0)) AS INT) AS free_kernels,
        CAST(ROUND(drink_savings   * disc_ml      / NULLIF(drink_full_price, 0))   AS INT) AS free_drink_ml
    FROM disc
);

-- Column: Premium experience viewings (Wrapped definition: anything not Regular)
CREATE OR REPLACE TEMPORARY VIEW col_premium_share AS (
    SELECT
        cde_id,
        SUM(CASE WHEN final_format <> 'Regular' THEN 1 ELSE 0 END) AS premium_viewings
    FROM cc_viewing_experience
    GROUP BY cde_id
);

-- Columns: Viewings by premium experience
CREATE OR REPLACE TEMPORARY VIEW col_format_counts AS (
    SELECT
        cde_id,
        SUM(CASE WHEN final_format = 'ThreeD'  THEN 1 ELSE 0 END) AS ThreeD_viewings,
        SUM(CASE WHEN final_format = 'DBOX'    THEN 1 ELSE 0 END) AS DBOX_viewings,
        SUM(CASE WHEN final_format = '70MM'    THEN 1 ELSE 0 END) AS MM70_viewings,
        SUM(CASE WHEN final_format = 'AVX'     THEN 1 ELSE 0 END) AS AVX_viewings,
        SUM(CASE WHEN final_format = 'IMAX'    THEN 1 ELSE 0 END) AS IMAX_viewings,
        SUM(CASE WHEN final_format = 'VIP'     THEN 1 ELSE 0 END) AS VIP_viewings,
        SUM(CASE WHEN final_format = 'FourDX'  THEN 1 ELSE 0 END) AS FourDX_viewings,
        SUM(CASE WHEN final_format = 'ScreenX' THEN 1 ELSE 0 END) AS ScreenX_viewings
    FROM cc_viewing_experience
    GROUP BY cde_id
);


-- Top 10 highest-grossing titles of 2026 (all guests, box office, Jan 1 → yesterday)
CREATE OR REPLACE TEMPORARY VIEW cc_top10_grossing AS (
    SELECT title_id, title_name, box_office_gross, gross_rank
    FROM (
        SELECT
            f.title_id,
            MAX(f.title_name)                                AS title_name,
            SUM(t.Net_Revenue)                               AS box_office_gross,
            ROW_NUMBER() OVER (ORDER BY SUM(t.Net_Revenue) DESC) AS gross_rank
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction AS t
        INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_film AS f
            ON t.film_id = f.film_id
        WHERE t.TicketFLAG = 'Yes'
          AND t.VisitDateId BETWEEN '20260101' AND date_format(curdate() - 1, 'yyyyMMdd')
        GROUP BY f.title_id
    ) AS ranked
    WHERE gross_rank <= 10
);

-- Blockbuster: distinct top-10 titles each member saw
CREATE OR REPLACE TEMPORARY VIEW col_blockbuster AS (
    SELECT
        v.cde_id,
        COUNT(DISTINCT v.title_id) AS blockbuster_titles_seen
    FROM cc_viewings AS v
    INNER JOIN cc_top10_grossing AS g
        ON v.title_id = g.title_id
    GROUP BY v.cde_id
);

-- Week one: first viewing of each title vs that title's release date
CREATE OR REPLACE TEMPORARY VIEW col_week_one AS (
    WITH first_seen AS (
        SELECT cde_id, title_id, MIN(visitdateid) AS first_visit
        FROM cc_viewings
        WHERE title_id IS NOT NULL
        GROUP BY cde_id, title_id
    ),
    title_release AS (
        SELECT title_id, MIN(theatrical_release_date) AS release_date
        FROM cpx_dataanalytics_gold.customer360.cpx_d_film
        GROUP BY title_id
    )
    SELECT
        fs.cde_id,
        COUNT(*) AS distinct_titles,
        SUM(CASE WHEN datediff(to_date(CAST(fs.first_visit AS STRING), 'yyyyMMdd'), r.release_date)
                      BETWEEN -5 AND 6 THEN 1 ELSE 0 END) AS week_one_titles
    FROM first_seen AS fs
    LEFT JOIN title_release AS r
        ON fs.title_id = r.title_id
    GROUP BY fs.cde_id
);

-- Badges (supporting counts kept here for checking; only Y/N go to the output)
CREATE OR REPLACE TEMPORARY VIEW col_badges AS (
    WITH visits AS (
        SELECT
            w.cde_id,
            COALESCE(tv.total_visits, 0)      AS total_visits,
            COALESCE(cv.concession_visits, 0) AS concession_visits
        FROM wrapped_members AS w
        LEFT JOIN col_total_visits      AS tv ON w.cde_id = tv.cde_id
        LEFT JOIN col_concession_visits AS cv ON w.cde_id = cv.cde_id
    ),
    cutoff AS (
        SELECT percentile_approx(total_visits, 0.9) AS p90_visits
        FROM visits
    )
    SELECT
        v.cde_id,
        c.p90_visits,
        COALESCE(b.blockbuster_titles_seen, 0)                            AS blockbuster_titles_seen,
        wk.week_one_titles / NULLIF(wk.distinct_titles, 0)               AS week_one_ratio,
        (CASE WHEN fc.ThreeD_viewings  > 0 THEN 1 ELSE 0 END
       + CASE WHEN fc.DBOX_viewings    > 0 THEN 1 ELSE 0 END
       + CASE WHEN fc.MM70_viewings    > 0 THEN 1 ELSE 0 END
       + CASE WHEN fc.AVX_viewings     > 0 THEN 1 ELSE 0 END
       + CASE WHEN fc.IMAX_viewings    > 0 THEN 1 ELSE 0 END
       + CASE WHEN fc.VIP_viewings     > 0 THEN 1 ELSE 0 END
       + CASE WHEN fc.FourDX_viewings  > 0 THEN 1 ELSE 0 END
       + CASE WHEN fc.ScreenX_viewings > 0 THEN 1 ELSE 0 END)          AS experience_types_used,

        CASE WHEN v.total_visits > c.p90_visits                           THEN 'Y' ELSE 'N' END AS Top_10pct_Visits_Badge,
        CASE WHEN v.total_visits > 0
              AND v.concession_visits / v.total_visits >= 0.75            THEN 'Y' ELSE 'N' END AS Concession_75pct_Badge,
        CASE WHEN COALESCE(b.blockbuster_titles_seen, 0) >= 5             THEN 'Y' ELSE 'N' END AS Blockbuster_Badge,
        CASE WHEN wk.week_one_titles / NULLIF(wk.distinct_titles, 0) > 0.5 THEN 'Y' ELSE 'N' END AS Week_One_Badge,
        CASE WHEN (CASE WHEN fc.ThreeD_viewings  > 0 THEN 1 ELSE 0 END
                 + CASE WHEN fc.DBOX_viewings    > 0 THEN 1 ELSE 0 END
                 + CASE WHEN fc.MM70_viewings    > 0 THEN 1 ELSE 0 END
                 + CASE WHEN fc.AVX_viewings     > 0 THEN 1 ELSE 0 END
                 + CASE WHEN fc.IMAX_viewings    > 0 THEN 1 ELSE 0 END
                 + CASE WHEN fc.VIP_viewings     > 0 THEN 1 ELSE 0 END
                 + CASE WHEN fc.FourDX_viewings  > 0 THEN 1 ELSE 0 END
                 + CASE WHEN fc.ScreenX_viewings > 0 THEN 1 ELSE 0 END) >= 4 THEN 'Y' ELSE 'N' END AS Experience_Badge
    FROM visits AS v
    CROSS JOIN cutoff AS c
    LEFT JOIN col_blockbuster   AS b  ON v.cde_id = b.cde_id
    LEFT JOIN col_week_one      AS wk ON v.cde_id = wk.cde_id
    LEFT JOIN col_format_counts AS fc ON v.cde_id = fc.cde_id
);

-- Columns: Seen flag per top-10 grossing film (Movie_1 = #1 by gross)
CREATE OR REPLACE TEMPORARY VIEW col_top10_flags AS (
    WITH seen AS (
        SELECT DISTINCT v.cde_id, g.gross_rank
        FROM cc_viewings AS v
        INNER JOIN cc_top10_grossing AS g
            ON v.title_id = g.title_id
    )
    SELECT
        w.cde_id,
        MAX(CASE WHEN s.gross_rank = 1  THEN 'Y' ELSE 'N' END) AS Movie_1_BB_Flag,
        MAX(CASE WHEN s.gross_rank = 2  THEN 'Y' ELSE 'N' END) AS Movie_2_BB_Flag,
        MAX(CASE WHEN s.gross_rank = 3  THEN 'Y' ELSE 'N' END) AS Movie_3_BB_Flag,
        MAX(CASE WHEN s.gross_rank = 4  THEN 'Y' ELSE 'N' END) AS Movie_4_BB_Flag,
        MAX(CASE WHEN s.gross_rank = 5  THEN 'Y' ELSE 'N' END) AS Movie_5_BB_Flag,
        MAX(CASE WHEN s.gross_rank = 6  THEN 'Y' ELSE 'N' END) AS Movie_6_BB_Flag,
        MAX(CASE WHEN s.gross_rank = 7  THEN 'Y' ELSE 'N' END) AS Movie_7_BB_Flag,
        MAX(CASE WHEN s.gross_rank = 8  THEN 'Y' ELSE 'N' END) AS Movie_8_BB_Flag,
        MAX(CASE WHEN s.gross_rank = 9  THEN 'Y' ELSE 'N' END) AS Movie_9_BB_Flag,
        MAX(CASE WHEN s.gross_rank = 10 THEN 'Y' ELSE 'N' END) AS Movie_10_BB_Flag
    FROM wrapped_members AS w
    LEFT JOIN seen AS s
        ON w.cde_id = s.cde_id
    GROUP BY w.cde_id
);

-- Column: Ticket credits left ($9.99 + $10.99, yesterday's snapshot)
CREATE OR REPLACE TEMPORARY VIEW col_credits_left AS (
    SELECT
        a.cde_id,
        COALESCE(s.recognitions_left, 0)                                         AS credits_left_999,
        COALESCE(s.recognitions_left_10_99, 0)                                   AS credits_left_1099,
        COALESCE(s.recognitions_left, 0) + COALESCE(s.recognitions_left_10_99, 0) AS ticket_credits_left
    FROM active_CC_Users AS a
    INNER JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot AS s
        ON  a.cc_user_profile_id = s.cc_user_profile_id
        AND to_date(s.snapshot_date_id, 'yyyyMMdd') = curdate() - 1
);

-- F&B mapping file (Wrapped_FnB_2026.xlsx) read from the Volume
CREATE OR REPLACE TEMPORARY VIEW fnb_raw AS
SELECT *
FROM read_files(
    '/Volumes/data_exploration_hub/marketing_hub/offline_data/Wrapped_FnB_2026.xlsx',
    format => 'excel',
    headerRows => 1
);

-- F&B mapping additions (hardcoded): 233 codes missing from the xlsx, matched to its categories
-- App (14xxx) drinks/candy: flavour lines mapped, parent lines left out (14067, 14068, 14115, 14116,
-- 14081, 14143, 14085, 14144) so each purchase counts once. Coffee/tea/smoothies have no category.
CREATE OR REPLACE TEMPORARY VIEW fnb_item_map_add AS
SELECT * FROM VALUES
    ('92827', 'Beer'),
    ('97093', 'Burger'),
    ('97094', 'Burger'),
    ('14010', 'Candy'),
    ('14012', 'Candy'),
    ('14014', 'Candy'),
    ('14019', 'Candy'),
    ('14021', 'Candy'),
    ('14022', 'Candy'),
    ('14024', 'Candy'),
    ('14082', 'Candy'),
    ('14083', 'Candy'),
    ('14156', 'Candy'),
    ('14157', 'Candy'),
    ('14158', 'Candy'),
    ('250390', 'Candy'),
    ('260373', 'Candy'),
    ('260804', 'Candy'),
    ('261042', 'Candy'),
    ('261584', 'Candy'),
    ('88121', 'Candy'),
    ('88123', 'Candy'),
    ('88210', 'Candy'),
    ('88357', 'Candy'),
    ('88727', 'Candy'),
    ('88848', 'Candy'),
    ('88850', 'Candy'),
    ('89371', 'Candy'),
    ('89498', 'Candy'),
    ('93280', 'Candy'),
    ('96142', 'Candy'),
    ('96144', 'Candy'),
    ('99639', 'Candy'),
    ('14011', 'Chocolate'),
    ('14013', 'Chocolate'),
    ('14015', 'Chocolate'),
    ('14016', 'Chocolate'),
    ('14017', 'Chocolate'),
    ('14018', 'Chocolate'),
    ('14020', 'Chocolate'),
    ('14023', 'Chocolate'),
    ('14026', 'Chocolate'),
    ('14027', 'Chocolate'),
    ('14028', 'Chocolate'),
    ('14029', 'Chocolate'),
    ('14146', 'Chocolate'),
    ('14159', 'Chocolate'),
    ('250380', 'Chocolate'),
    ('250385', 'Chocolate'),
    ('250386', 'Chocolate'),
    ('250387', 'Chocolate'),
    ('260189', 'Chocolate'),
    ('261261', 'Chocolate'),
    ('82462', 'Chocolate'),
    ('88849', 'Chocolate'),
    ('88851', 'Chocolate'),
    ('93282', 'Chocolate'),
    ('93966', 'Chocolate'),
    ('95810', 'Chocolate'),
    ('96143', 'Chocolate'),
    ('250042', 'Cocktail'),
    ('260831', 'Cocktail'),
    ('260832', 'Cocktail'),
    ('80269', 'Cocktail'),
    ('80558', 'Cocktail'),
    ('81306', 'Cocktail'),
    ('81782', 'Cocktail'),
    ('82309', 'Cocktail'),
    ('94575', 'Cocktail'),
    ('94776', 'Cocktail'),
    ('97583', 'Cocktail'),
    ('80402', 'Dessert'),
    ('80855', 'Dessert'),
    ('81102', 'Dessert'),
    ('81688', 'Dessert'),
    ('94209', 'Dessert'),
    ('94399', 'Dessert'),
    ('96724', 'Dessert'),
    ('96725', 'Dessert'),
    ('80382', 'Finger Food'),
    ('88544', 'Finger Food'),
    ('89009', 'Finger Food'),
    ('89327', 'Finger Food'),
    ('89329', 'Finger Food'),
    ('89365', 'Finger Food'),
    ('93964', 'Finger Food'),
    ('94625', 'Finger Food'),
    ('95673', 'Finger Food'),
    ('95830', 'Finger Food'),
    ('95831', 'Finger Food'),
    ('95835', 'Finger Food'),
    ('95840', 'Finger Food'),
    ('96930', 'Finger Food'),
    ('96933', 'Finger Food'),
    ('97099', 'Finger Food'),
    ('98046', 'Finger Food'),
    ('14051', 'Hot Dog'),
    ('96924', 'Hot Dog'),
    ('250173', 'Milkshake'),
    ('81352', 'Milkshake'),
    ('88199', 'Milkshake'),
    ('88200', 'Milkshake'),
    ('88201', 'Milkshake'),
    ('89248', 'Milkshake'),
    ('96935', 'Milkshake'),
    ('14057', 'Nachos'),
    ('81990', 'Nachos'),
    ('96931', 'Nachos'),
    ('260458', 'Pizza'),
    ('260460', 'Pizza'),
    ('81985', 'Pizza'),
    ('82670', 'Pizza'),
    ('88746', 'Pizza'),
    ('96934', 'Pizza'),
    ('14069', 'Pop'),
    ('14070', 'Pop'),
    ('14071', 'Pop'),
    ('14072', 'Pop'),
    ('14073', 'Pop'),
    ('14074', 'Pop'),
    ('14075', 'Pop'),
    ('14076', 'Pop'),
    ('14077', 'Pop'),
    ('14078', 'Pop'),
    ('14079', 'Pop'),
    ('14091', 'Pop'),
    ('14092', 'Pop'),
    ('14093', 'Pop'),
    ('14094', 'Pop'),
    ('14095', 'Pop'),
    ('14096', 'Pop'),
    ('14097', 'Pop'),
    ('14098', 'Pop'),
    ('14099', 'Pop'),
    ('14100', 'Pop'),
    ('14101', 'Pop'),
    ('14117', 'Pop'),
    ('14118', 'Pop'),
    ('14119', 'Pop'),
    ('14120', 'Pop'),
    ('14121', 'Pop'),
    ('14122', 'Pop'),
    ('14123', 'Pop'),
    ('14124', 'Pop'),
    ('14125', 'Pop'),
    ('14126', 'Pop'),
    ('14127', 'Pop'),
    ('14128', 'Pop'),
    ('14129', 'Pop'),
    ('14130', 'Pop'),
    ('14131', 'Pop'),
    ('14132', 'Pop'),
    ('14133', 'Pop'),
    ('14134', 'Pop'),
    ('14135', 'Pop'),
    ('14136', 'Pop'),
    ('14137', 'Pop'),
    ('250109', 'Pop'),
    ('260707', 'Pop'),
    ('261469', 'Pop'),
    ('80082', 'Pop'),
    ('80084', 'Pop'),
    ('80094', 'Pop'),
    ('81526', 'Pop'),
    ('93262', 'Pop'),
    ('93970', 'Pop'),
    ('93976', 'Pop'),
    ('93977', 'Pop'),
    ('93980', 'Pop'),
    ('99862', 'Pop'),
    ('14000', 'Popcorn'),
    ('14001', 'Popcorn'),
    ('14084', 'Popcorn'),
    ('82343', 'Popcorn'),
    ('93847', 'Popcorn'),
    ('93848', 'Popcorn'),
    ('93849', 'Popcorn'),
    ('99025', 'Popcorn'),
    ('250467', 'Poutine'),
    ('96927', 'Poutine'),
    ('96928', 'Poutine'),
    ('89252', 'Sandwich'),
    ('14030', 'Water'),
    ('14031', 'Water'),
    ('14032', 'Water'),
    ('14034', 'Water'),
    ('14036', 'Water'),
    ('94070', 'Water'),
    ('99533', 'Water')
    -- Hot Beverages (new category, not in the xlsx): hot chocolate, coffee, tea
    ,('94250', 'Hot Beverages'),
    ('94251', 'Hot Beverages'),
    -- Coffee and tea (drip, VIP drip, decaf, Teavana/VIP tea, Starbucks hot espresso drinks)
    ('94157', 'Hot Beverages'),
    ('94158', 'Hot Beverages'),
    ('99358', 'Hot Beverages'),
    ('99359', 'Hot Beverages'),
    ('94136', 'Hot Beverages'),
    ('94137', 'Hot Beverages'),
    ('88698', 'Hot Beverages'),
    ('88699', 'Hot Beverages'),
    ('94263', 'Hot Beverages'),
    ('94161', 'Hot Beverages'),
    ('94162', 'Hot Beverages'),
    ('99360', 'Hot Beverages'),
    ('99361', 'Hot Beverages'),
    ('94241', 'Hot Beverages'),
    ('94242', 'Hot Beverages'),
    ('94151', 'Hot Beverages'),
    ('94246', 'Hot Beverages'),
    ('94247', 'Hot Beverages'),
    ('94239', 'Hot Beverages'),
    ('94240', 'Hot Beverages'),
    ('98951', 'Hot Beverages'),
    ('98950', 'Hot Beverages'),
    ('94248', 'Hot Beverages'),
    ('94249', 'Hot Beverages'),
    ('94206', 'Hot Beverages'),
    ('94243', 'Hot Beverages'),
    ('94244', 'Hot Beverages'),
    ('88696', 'Hot Beverages'),
    -- Popcorn coupons and combo coupons: the coupon line is the only record of the popcorn
    -- (popcorn lines in the same transaction are separately paid, ~$6.50)
    ('88395', 'Popcorn'),
    ('88396', 'Popcorn'),
    ('88183', 'Popcorn'),
    ('88193', 'Popcorn'),
    ('88191', 'Popcorn'),
    ('88190', 'Popcorn'),
    ('88185', 'Popcorn'),
    ('88187', 'Popcorn'),
    ('89566', 'Popcorn'),
    ('88186', 'Popcorn'),
    ('93862', 'Popcorn'),
    ('SM CORN', 'Popcorn'),
    ('88189', 'Popcorn'),
    ('88188', 'Popcorn')
AS t(item_code, final_category);

-- F&B mapping: xlsx (one row per Item_ID; dupes removed; conflicts by majority, e.g. 260584 -> Poutine)
--              + hardcoded additions for codes not in the xlsx
CREATE OR REPLACE TEMPORARY VIEW fnb_item_map AS (
    WITH src AS (
        SELECT
            TRIM(CAST(`Item Code` AS STRING)) AS item_code,
            TRIM(`Final Category`)            AS final_category
        FROM fnb_raw
        WHERE `Item Code` IS NOT NULL
    ),
    votes AS (
        SELECT item_code, final_category, COUNT(*) AS n
        FROM src
        GROUP BY item_code, final_category
    ),
    file_map AS (
        SELECT item_code, final_category
        FROM votes
        QUALIFY ROW_NUMBER() OVER (PARTITION BY item_code ORDER BY n DESC, final_category) = 1
    )
    SELECT item_code, final_category FROM file_map
    UNION ALL
    SELECT a.item_code, a.final_category
    FROM fnb_item_map_add AS a
    LEFT ANTI JOIN file_map AS f
        ON a.item_code = f.item_code
);

-- Line level: theatre F&B lines matched on cpx_d_item.Item_ID
-- Combo tags excluded: $0 marker lines; their contents are rung separately
CREATE OR REPLACE TEMPORARY VIEW cc_fnb_lines AS (
    SELECT
        t.cde_id,
        t.visitdateid,
        t.Item_ID,
        t.item_description,
        m.final_category,
        t.Quantity
    FROM cc_all_trans AS t
    INNER JOIN fnb_item_map AS m
        ON TRIM(CAST(t.Item_ID AS STRING)) = m.item_code
    WHERE t.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.STOREFLAG = 'No'
      AND m.final_category NOT IN ('Combo One', 'Combo Two', 'Combo Three', 'Mini Combo')
);

-- Columns: Item_1..Item_3 = top 3 F&B categories by net quantity
--          (tie -> most recently bought -> alphabetical)
CREATE OR REPLACE TEMPORARY VIEW col_top3_fnb AS (
    WITH category_qty AS (
        SELECT
            cde_id,
            final_category,
            SUM(Quantity)    AS net_qty,
            MAX(visitdateid) AS last_bought
        FROM cc_fnb_lines
        GROUP BY cde_id, final_category
        HAVING SUM(Quantity) > 0
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY cde_id
                ORDER BY net_qty DESC, last_bought DESC, final_category
            ) AS rn
        FROM category_qty
    )
    SELECT
        cde_id,
        MAX(CASE WHEN rn = 1 THEN final_category END) AS Item_1,
        MAX(CASE WHEN rn = 2 THEN final_category END) AS Item_2,
        MAX(CASE WHEN rn = 3 THEN final_category END) AS Item_3
    FROM ranked
    WHERE rn <= 3
    GROUP BY cde_id
);

-- =====================================================================
-- Final output: one row per member, one LEFT JOIN per column view
-- =====================================================================
CREATE OR REPLACE TEMPORARY VIEW wrapped_output AS (
    SELECT
        a.cc_user_profile_id,
        w.cde_id,
        a.plan_id,
        a.renewal_status,
        c1.joined_date,
        COALESCE(c2.total_visits, 0)            AS total_visits,
        COALESCE(c3.movie_visits, 0)            AS movie_visits,
        COALESCE(c4.concession_visits, 0)       AS concession_visits,
        COALESCE(c5.movies_watched, 0)          AS movies_watched,
        COALESCE(c5.unique_movies_watched, 0)   AS unique_movies_watched,
        COALESCE(c6.max_rewatch_count, 0)       AS max_rewatch_count,
        c6.max_rewatch_title_id,
        c7.titles_watched,
        COALESCE(c8.total_hours_watched, 0)     AS total_hours_watched,
        COALESCE(c9.ticket_savings_999, 0)      AS ticket_savings_999,
        COALESCE(c9.ticket_savings_1099, 0)     AS ticket_savings_1099,
        COALESCE(c9.concession_savings, 0)      AS concession_savings,
        COALESCE(c9.gaming_savings, 0)          AS gaming_savings,
        COALESCE(c9.online_booking_savings, 0)  AS online_booking_savings,
        COALESCE(c9.total_savings, 0)           AS total_savings,
        COALESCE(c10.Total_Kernels, 0)          AS Total_Kernels,
        COALESCE(c10.Total_Drink_ml, 0)         AS Total_Drink_ml,
        COALESCE(c11.free_kernels, 0)           AS free_kernels,
        COALESCE(c11.free_drink_ml, 0)          AS free_drink_ml,
        COALESCE(c12.premium_viewings, 0)       AS premium_viewings,
        COALESCE(c3.movie_visits, 0)
          - COALESCE(c12.premium_viewings, 0)   AS regular_viewings,
        COALESCE(c13.ThreeD_viewings, 0)        AS ThreeD_viewings,
        COALESCE(c13.DBOX_viewings, 0)          AS DBOX_viewings,
        COALESCE(c13.MM70_viewings, 0)          AS MM70_viewings,
        COALESCE(c13.AVX_viewings, 0)           AS AVX_viewings,
        COALESCE(c13.IMAX_viewings, 0)          AS IMAX_viewings,
        COALESCE(c13.VIP_viewings, 0)           AS VIP_viewings,
        COALESCE(c13.FourDX_viewings, 0)        AS FourDX_viewings,
        COALESCE(c13.ScreenX_viewings, 0)       AS ScreenX_viewings,
        c15.Top_10pct_Visits_Badge,
        c15.Concession_75pct_Badge,
        c15.Blockbuster_Badge,
        c15.Week_One_Badge,
        c15.Experience_Badge
        -- ,c16.Movie_1_BB_Flag
        -- ,c16.Movie_2_BB_Flag
        -- ,c16.Movie_3_BB_Flag
        -- ,c16.Movie_4_BB_Flag
        -- ,c16.Movie_5_BB_Flag
        -- ,c16.Movie_6_BB_Flag
        -- ,c16.Movie_7_BB_Flag
        -- ,c16.Movie_8_BB_Flag
        -- ,c16.Movie_9_BB_Flag
        -- ,c16.Movie_10_BB_Flag
        ,COALESCE(c17.ticket_credits_left, 0) AS ticket_credits_left
        ,c18.Item_1
        ,c18.Item_2
        ,c18.Item_3
    FROM wrapped_members AS w
    INNER JOIN active_CC_Users      AS a   ON w.cde_id = a.cde_id
    LEFT JOIN col_join_date         AS c1  ON w.cde_id = c1.cde_id
    LEFT JOIN col_total_visits      AS c2  ON w.cde_id = c2.cde_id
    LEFT JOIN col_movie_visits      AS c3  ON w.cde_id = c3.cde_id
    LEFT JOIN col_concession_visits AS c4  ON w.cde_id = c4.cde_id
    LEFT JOIN col_movies_watched    AS c5  ON w.cde_id = c5.cde_id
    LEFT JOIN col_max_rewatch       AS c6  ON w.cde_id = c6.cde_id
    LEFT JOIN col_titles_watched    AS c7  ON w.cde_id = c7.cde_id
    LEFT JOIN col_hours_watched     AS c8  ON w.cde_id = c8.cde_id
    LEFT JOIN col_savings           AS c9  ON w.cde_id = c9.cde_id
    LEFT JOIN col_pop_drink         AS c10 ON w.cde_id = c10.cde_id
    LEFT JOIN col_pop_drink_free    AS c11 ON w.cde_id = c11.cde_id
    LEFT JOIN col_premium_share     AS c12 ON w.cde_id = c12.cde_id
    LEFT JOIN col_format_counts     AS c13 ON w.cde_id = c13.cde_id
    LEFT JOIN col_badges            AS c15 ON w.cde_id = c15.cde_id
    LEFT JOIN col_top10_flags       AS c16 ON w.cde_id = c16.cde_id
    LEFT JOIN col_credits_left      AS c17 ON w.cde_id = c17.cde_id
    LEFT JOIN col_top3_fnb          AS c18 ON w.cde_id = c18.cde_id
);

SELECT * FROM wrapped_output;